# 05 · Entrenamiento del modelo v3 (con data augmentation)

**Objetivo:** entrenar la tercera versión del detector aplicando data augmentation sobre el dataset de 743 imágenes, con el fin de mejorar la robustez del modelo y reducir la confusión entre clases identificada en el modelo v2.

**Contexto:** el análisis del modelo v2 reveló confusión bidireccional entre árbitro y jugador, atribuible a la variabilidad cromática del árbitro. Para mitigarlo se aplican augmentaciones de color (brillo, exposición, tono, saturación) que reducen la dependencia del modelo respecto al color exacto.

**Entrada:** dataset versión 3 de Roboflow (743 imágenes + augmentation, ~2000 imágenes de entrenamiento).

**Salida:** modelo `best_v3.pt` y métricas.

## 1. Descarga del dataset aumentado desde Roboflow

Se descarga la versión 3 del dataset, que incluye las augmentaciones aplicadas únicamente sobre el conjunto de entrenamiento. Los conjuntos de validación y test permanecen sin aumentar, lo que garantiza una evaluación justa.

In [1]:
from roboflow import Roboflow
from getpass import getpass
from pathlib import Path
import shutil

REPO_ROOT = Path("../..").resolve()
DATASET_DIR = REPO_ROOT / "data" / "dataset_v3"

WORKSPACE = "hugos-workspace-0dwnk"
PROJECT = "football-tactical-analysis"

api_key = getpass("Introduce tu API key de Roboflow: ")

In [2]:
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
    print("Dataset v3 antiguo eliminado")

rf = Roboflow(api_key=api_key)
project = rf.workspace(WORKSPACE).project(PROJECT)

# Descargar la versión 3 (con augmentation)
version = project.version(3)
dataset = version.download("yolov8", location=str(DATASET_DIR))

print(f"\nDataset v3 descargado en: {DATASET_DIR}")

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 95.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset_v3 in yolov8:: 100%|██████████| 3571/3571 [00:03<00:00, 977.15it/s] 



Dataset v3 descargado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset_v3


## 2. Verificación del dataset aumentado

In [3]:
import torch
from ultralytics import YOLO

DATA_YAML = DATASET_DIR / "data.yaml"

train_imgs = list((DATASET_DIR / "train" / "images").glob("*.jpg"))
valid_imgs = list((DATASET_DIR / "valid" / "images").glob("*.jpg"))
test_imgs = list((DATASET_DIR / "test" / "images").glob("*.jpg")) if (DATASET_DIR / "test").exists() else []

print(f"Train: {len(train_imgs)} imágenes (con augmentation)")
print(f"Valid: {len(valid_imgs)} imágenes")
print(f"Test:  {len(test_imgs)} imágenes")
print(f"Total: {len(train_imgs) + len(valid_imgs) + len(test_imgs)} imágenes")
print(f"\nGPU disponible: {torch.cuda.is_available()}")

Train: 1560 imágenes (con augmentation)
Valid: 149 imágenes
Test:  74 imágenes
Total: 1783 imágenes

GPU disponible: True


## 3. Corrección del data.yaml

In [4]:
test_line = f"test: {DATASET_DIR}/test/images\n" if test_imgs else ""

yaml_content = f"""train: {DATASET_DIR}/train/images
val: {DATASET_DIR}/valid/images
{test_line}nc: 3
names: ['ball', 'player', 'referee']
"""

with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("data.yaml actualizado:\n")
print(yaml_content)

data.yaml actualizado:

train: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset_v3/train/images
val: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset_v3/valid/images
test: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset_v3/test/images
nc: 3
names: ['ball', 'player', 'referee']



## 4. Entrenamiento del modelo v3

Se mantienen los mismos hiperparámetros que en v1 y v2 (50 épocas, 640px, batch 16) para que la única variable diferencial sea el data augmentation. Esto permite aislar su efecto sobre el rendimiento.

Nota: se desactivan las augmentaciones internas de YOLO (`augment=False` implícito en estos parámetros) porque el aumento de datos ya se ha realizado en Roboflow, evitando duplicar el efecto.

In [5]:
model = YOLO("yolov8s.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project=str(REPO_ROOT / "runs" / "detect" / "models"),
    name="detector_v3",
    patience=15,
    save=True,
    plots=True,
    workers=0
)

print(f"\nEntrenamiento completado")
print(f"Resultados guardados en: {results.save_dir}")

Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset_v3\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=detector_v3, nbs=64, nms=False, opset=None, optimize

## 5. Métricas del modelo v3

In [6]:
metrics = model.val(workers=0)

print("\nMétricas finales del modelo v3:\n")
print(f"{'Clase':<15} {'Precisión':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print("-" * 60)

for i, cls in model.names.items():
    p = metrics.box.p[i]
    r = metrics.box.r[i]
    map50 = metrics.box.ap50[i]
    map5095 = metrics.box.ap[i]
    print(f"{cls:<15} {p:>10.3f} {r:>10.3f} {map50:>10.3f} {map5095:>10.3f}")

print("-" * 60)
print(f"{'Promedio':<15} {metrics.box.mp:>10.3f} {metrics.box.mr:>10.3f} {metrics.box.map50:>10.3f} {metrics.box.map:>10.3f}")

Ultralytics 8.4.53  Python-3.13.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 368.6110.6 MB/s, size: 56.6 KB)
val: Scanning C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\dataset_v3\valid\labels.cache... 149 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 149/149 32.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.3it/s 4.4s.3ss
                   all        149       3439      0.879      0.784      0.815      0.491
                  ball        125        125      0.802      0.584      0.597      0.209
                player        149       3012      0.939      0.971      0.972      0.706
               referee        143        302      0.896      0.797      0.876      0.557
Speed: 1.4ms preprocess, 15.1ms inference, 0.0ms loss, 1.6ms postprocess per ima

## 6. Guardado del modelo v3

In [7]:
source = Path(results.save_dir) / "weights" / "best.pt"
destination = REPO_ROOT / "models" / "best_v3.pt"

shutil.copy2(source, destination)

print(f"Modelo v3 guardado en: {destination}")
print(f"Tamaño: {destination.stat().st_size / (1024*1024):.2f} MB")

Modelo v3 guardado en: C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\models\best_v3.pt
Tamaño: 21.48 MB


## 7. Conclusiones

Tras este notebook:

- Se ha entrenado el modelo v3 sobre el dataset aumentado con variaciones de color y transformaciones geométricas suaves.
- El modelo queda guardado como `models/best_v3.pt`.
- La comparación rigurosa entre v1, v2 y v3 sobre un conjunto de test común se realiza en el notebook de comparativa final, donde se evaluará si el data augmentation ha reducido efectivamente la confusión árbitro-jugador y mejorado la detección del balón.